# Task 1 — Data Understanding
## Distributed Flight Cancellation Risk Prediction Using PySpark and US Airline Operational Data

### 1. Project Objective & Problem Definition
The primary business objective of this project is to develop a distributed machine learning solution using PySpark that predicts whether a scheduled flight will be cancelled before departure. Flight cancellations cause massive operational disruptions for airlines, cost airports millions in passenger recovery services, and leave passengers stranded. 

By predicting flight cancellation risks in advance:
1. **Airlines** can optimize crew scheduling, aircraft allocation, and reserve management.
2. **Airport Operators** can proactively manage gate availability, security staffing, and passenger flow.
3. **Passengers** can be notified early to rebook, minimizing travel anxiety and customer service bottlenecks.

This is modeled as a **binary classification problem** because the target variable is discrete and contains two possible states:
- **`Cancelled = 1.0`**: Flight Cancelled before departure.
- **`Cancelled = 0.0`**: Flight Operated.

Early prediction is highly valuable for disruptive-event forecasting, and distributed processing (Spark) is critical because airline datasets easily scale to hundreds of millions of records annually.



### 2. Dataset Setup and Inspection
First, we set the environment variable for the Java Security Manager issue with Java 18+ and start our optimized Spark Session.



In [ ]:
import os
import sys
import ctypes

def get_short_path(long_name):
    try:
        buf = ctypes.create_unicode_buffer(1024)
        ctypes.windll.kernel32.GetShortPathNameW(long_name, buf, 1024)
        return buf.value if buf.value else long_name
    except Exception:
        return long_name

# Workaround for Java 18+ Security Manager issue on Windows
os.environ['SPARK_SUBMIT_OPTS'] = '-Djava.security.manager=allow'
os.environ['HADOOP_HOME'] = get_short_path(r'C:\hadoop')

python_exe = get_short_path(sys.executable)
os.environ['PYSPARK_PYTHON'] = python_exe
os.environ['PYSPARK_DRIVER_PYTHON'] = python_exe

java_home = os.environ.get('JAVA_HOME', '')
if not java_home or not os.path.exists(os.path.join(java_home, 'bin', 'java.exe')):
    default_java = r'C:\Program Files\Eclipse Adoptium\jdk-17.0.16.8-hotspot'
    if os.path.exists(os.path.join(default_java, 'bin', 'java.exe')):
        os.environ['JAVA_HOME'] = get_short_path(default_java)
import time
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Initialize Spark Session optimized for 8GB RAM, Ryzen 5 5500H
spark = SparkSession.builder \
    .appName("Task1_Data_Understanding") \
    .master("local[4]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "12") \
    .getOrCreate()

print("Spark Session Created Successfully!")



### 3. Load and Preview Dataset


In [ ]:
# Load the dataset
csv_path = "../../PR/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv"
df = spark.read.csv(csv_path, header=True, inferSchema=True)

# Preview first 20 records
df.show(20)



### 4. Dataset Statistics and Schema Description


In [ ]:
# Display schema
df.printSchema()



In [ ]:
# Dataset dimensions
total_rows = df.count()
total_cols = len(df.columns)
file_size_mb = os.path.getsize(csv_path) / (1024 * 1024)
partition_count = df.rdd.getNumPartitions()

print(f"Total Rows: {total_rows}")
print(f"Total Columns: {total_cols}")
print(f"Dataset File Size: {file_size_mb:.2f} MB")
print(f"Storage Format: CSV")
print(f"Number of Partitions: {partition_count}")



### 5. Five Vs of Big Data for Airline Operational Data
| Big Data V | Airline Operational Data Specifics | Project-specific Application |
|---|---|---|
| **Volume** | Millions of flight records are generated monthly by BTS, tracking flight dates, airlines, times, and delays. | The dataset size is **235.40 MB** containing **547,271** flights for a single month. This scales exponentially for multi-year analyses. |
| **Velocity** | Flight tracking systems and IoT devices on aircraft generate real-time feeds of tail movements, pushbacks, and status updates. | Spark's distributed processing allows high-speed batch execution and streaming ingestion of flights. |
| **Variety** | Data includes categorical values (Airlines, Origins, Destinations), temporal factors, numeric metrics (Distance, Taxi Times), and text (cancellation codes). | Handled by PySpark indexing, scaling, and OHE pipelines. |
| **Veracity** | Noise exists due to missing values (nulls in taxi times for cancelled flights) and manual input inconsistencies. | Imputing `TaxiOut` to 0.0 and validating missing values maintains schema integrity. |
| **Value** | Anticipating cancellations allows airlines to reallocate planes, save millions in gate fees, and preserve brand reputation. | The classification model identifies flight-specific cancel indicators to guide business decisions. |



### 6. Ethical and Licensing Considerations
- **Public Transport & Passenger Privacy**: The dataset is public data released by the US Bureau of Transportation Statistics. It contains no Personally Identifiable Information (PII) of passengers, making it compliant with privacy policies (e.g., GDPR, CCPA).
- **Bias Concerns**: Predictions might be biased towards major airlines or hubs with higher traffic (e.g., Atlanta, Chicago). A model might overpredict cancellations at smaller hubs due to sparse data.
- **Responsible Use**: Model decisions should not automatically blacklist specific flights or carriers but should be used for resource planning and proactive routing suggestions.



### 7. Class Distribution Visualisation


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plot pre-generated class distribution chart
# (or read from df to show locally)
class_counts = df.groupBy("Cancelled").count().toPandas()
print(class_counts)

plt.figure(figsize=(6, 4))
sns.barplot(x="Cancelled", y="count", data=class_counts, palette="viridis")
plt.title("Flight Status Distribution (0 = Operated, 1 = Cancelled)")
plt.xlabel("Cancelled")
plt.ylabel("Number of Flights")
plt.tight_layout()
plt.show()



### 8. Verification & Screenshots Checklist
1. **Dataset Preview**: Confirm the outputs of `df.show(20)` are correctly aligned.
2. **Schema Description**: Ensure columns like `Cancelled`, `TaxiOut`, and others are typed as numeric.
3. **File Size**: Check if Python path matches and correctly shows the file size.
4. **Dataset Statistics**: Ensure total rows match `547,271`.



In [ ]:
# Stop Spark session
spark.stop()

